<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0">

# Stochastic Short Rates

This brief section illustrates the use of stochastic short rate models for simulation and (risk-neutral) discounting. The class used is called `stochastic_short_rate`.

## The Modelling

First, the market environment. As a stochastic short rate model the `square_root_diffusion` class is (currently) available. We therefore need to define the respective parameters for this class in the market environment.

In [ ]:
!git clone https://github.com/tpq-classes/computational_finance_dx.git
import sys
sys.path.append('computational_finance_dx')

!git clone https://github.com/yhilpisch/dx.git
sys.path.append('dx')


In [ ]:
from dx import *

In [ ]:
me = market_environment(name='me', pricing_date=dt.datetime(2020, 1, 1))
me.add_constant('initial_value', 0.01)
me.add_constant('volatility', 0.1)
me.add_constant('kappa', 2.0)
me.add_constant('theta', 0.05)
me.add_constant('paths', 1000)
me.add_constant('frequency', 'M')
me.add_constant('starting_date', me.pricing_date)
me.add_constant('final_date', dt.datetime(2020, 12, 31))
me.add_curve('discount_curve', 0.0)  # dummy
me.add_constant('currency', 0.0)  # dummy

Second, the instantiation of the class.

In [ ]:
ssr = stochastic_short_rate('sr', me)

The following is an example `list` object containing `datetime` objects.

In [ ]:
time_list = [dt.datetime(2020, 1, 1),
             dt.datetime(2020, 4, 1),
             dt.datetime(2020, 6, 15),
             dt.datetime(2020, 10, 21)]

The call of the method `get_forward_reates()` yields the above `time_list` object and the simulated forward rates. In this case, 10 simulations.

In [ ]:
ssr.get_forward_rates(time_list, 10)

Accordingly, the call of the `get_discount_factors()` method yields simulated zero-coupon bond prices for the time grid.

In [ ]:
ssr.get_discount_factors(time_list, 10)

## Stochastic Drifts

Let us use the stochastic short rate model to simulate a geometric Brownian motion with stochastic short rate. Define the market environment as follows:

The Black-Scholes-Merton (1973) SDE becomes:

$$dS_t = r_t   S_t  dt + \sigma  S_t dZ_t$$

In [ ]:
me.add_constant('initial_value', 36.)
me.add_constant('volatility', 0.2)
  # time horizon for the simulation
me.add_constant('currency', 'EUR')
me.add_constant('frequency', 'M')
  # monthly frequency; paramter accoring to pandas convention
me.add_constant('paths', 10000)
  # number of paths for simulation

Then add the `stochastic_short_rate` object as discount curve.

In [ ]:
me.add_curve('discount_curve', ssr)

Finally, instantiate the `geometric_brownian_motion` object.

In [ ]:
gbm = geometric_brownian_motion('gbm', me)

We get simulated instrument values as usual via the `get_instrument_values()` method.

In [ ]:
gbm.get_instrument_values()[:, :4]

In [ ]:
from pylab import plt
plt.style.use('seaborn-v0_8')
%matplotlib inline

In [ ]:
# risk factor paths
plt.figure(figsize=(10, 6))
plt.plot(gbm.instrument_values[:, :25], '--')
plt.plot(gbm.instrument_values.mean(axis=1), 'r-', lw=3.5);

In [ ]:
# short rate paths
plt.figure(figsize=(10, 6))
plt.plot(ssr.process.instrument_values[:, :25], '--')
plt.plot(ssr.process.instrument_values.mean(axis=1), 'r-', lw=3.5);

## Valuation with Stochastic Short Rate

Let us compare the valuation of an American put option with a constant short rate and a stochastic one.

### Constant Short Rate

In [ ]:
csr = constant_short_rate('r', 0.06)

In [ ]:
me.add_constant('initial_value', 36.)
me.add_constant('volatility', 0.2)
me.add_constant('maturity', dt.datetime(2020, 12, 31))
me.add_constant('strike', 40.)
me.add_constant('paths', 50000)
me.add_curve('discount_curve', csr)

In [ ]:
gbm = geometric_brownian_motion('gbm', me)

In [ ]:
put_ame = valuation_mcs_american_single(
                name='put_ame',
                underlying=gbm,
                mar_env=me,
                payoff_func='np.maximum(strike - instrument_values, 0)')

In [ ]:
put_ame.present_value()

### Stochastic Short Rate

In [ ]:
me.add_curve('discount_curve', ssr)

In [ ]:
gbm = geometric_brownian_motion('gbm', me)

In [ ]:
put_ame = valuation_mcs_american_single(
                name='put_ame',
                underlying=gbm,
                mar_env=me,
                payoff_func='np.maximum(strike - instrument_values, 0)')

In [ ]:
put_ame.present_value()

**Copyright, License & Disclaimer**

© Dr. Yves J. Hilpisch | The Python Quants GmbH

DX Analytics (the "dx library" or "dx package") is licensed under the GNU Affero General
Public License version 3 or later (see http://www.gnu.org/licenses/).

DX Analytics comes with no representations or warranties, to the extent
permitted by applicable law.

http://tpq.io | [dx@tpq.io](mailto:team@tpq.io) |
http://twitter.com/dyjh

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

**Quant Platform** | http://pqp.io

**Python for Finance Training** | http://training.tpq.io

**Certificate in Computational Finance** | http://compfinance.tpq.io

**Derivatives Analytics with Python (Wiley Finance)** |
http://dawp.tpq.io

**Python for Finance (2nd ed., O'Reilly)** |
http://py4fi.tpq.io